<a href="https://colab.research.google.com/github/Wai-Fun/Healthcare-Data-Analytics/blob/main/Project_Healthcare_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project: Healthcare Data Analysis with Python and SQL**




In [553]:
# Install required packages
import sqlite3
import requests
import pandas as pd
from pathlib import Path

In [554]:
#load dataset
df = pd.read_csv("https://foundations-of-healthcare-data-analytics-4e579d.gitlab.io/labs/Projects/Patients_EHR_raw_data.csv")

###**Data Cleaning and EDA Using Python**

##0. Dataset preview and basic information

In [555]:
# Preview dataset df
df.head(20)

,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
0,NV001,1983-03-11,Female,30933.0,Type 2 Diabetes (E11),07-06-2024
1,NV004,NaN,Male,76999.0,Asthma (J45.909),27-10-2022
2,NV005,1984-08-25,Female,40307.0,Osteoarthritis (M15.9),30-10-2024
3,NV006,1982-12-16,Female,56844.0,Type 2 Diabetes (E11),05-07-2022
4,NV007,1990-10-11,Male,NaN,Hyperlipidemia (E78.5),17-06-2024
5,NV008,1949-07-08,Male,56844.0,Hyperlipidemia (E78.5),45258
6,NV009,1955-07-11,Female,68045.0,Hyperlipidemia (E78.5),09-05-2024
7,NV010,1944-11-11,Female,41617.0,Osteoarthritis (M15.9),25-06-2023
8,NV012,1975-01-14,Female,33023.0,Hypertension (I10),21-09-2024
9,NV014,1943-08-25,Female,26372.0,Hyperlipidemia (E78.5),07-07-2022


In [556]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         74 non-null     object 
 1   DateOfBirth       71 non-null     object 
 2   Gender            74 non-null     object 
 3   ZipCode           70 non-null     float64
 4   PrimaryCondition  74 non-null     object 
 5   AdmissionDate     74 non-null     object 
dtypes: float64(1), object(5)
memory usage: 3.6+ KB


### 1a. Identify Missing Values

In [557]:

# Count number of rows with missing values
missing_row_count = df.isnull().any(axis=1).sum()
print(f"Number of rows with missing values: {missing_row_count} \n")

# List all rows that contain any missing value
df[df.isnull().any(axis=1)]

Number of rows with missing values: 7 



,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
1,NV004,NaN,Male,76999.0,Asthma (J45.909),27-10-2022
4,NV007,1990-10-11,Male,NaN,Hyperlipidemia (E78.5),17-06-2024
20,NV028,18-05-1978,Male,NaN,Type 2 Diabetes (E11),27-10-2022
21,NV028,18-05-1978,Male,NaN,Type 2 Diabetes (E11),27-10-2022
33,NV043,1970-07-08,Male,NaN,Hypertension (I10),09-03-2024
47,NV059,NaN,Female,53017.0,UNKNOWN,15-12-2023
67,NV088,NaN,Female,37775.0,Osteoarthritis (M15.9),02-09-2022


### 1b. Identify duplicate rows

In [558]:
# Count number of duplicated rows
duplicate_row_count= df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_row_count} \n")

# List all rows that contain duplicate rows
df[df.duplicated()]

Number of duplicate rows: 2 



,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
21,NV028,18-05-1978,Male,NaN,Type 2 Diabetes (E11),27-10-2022
44,NV053,1941-04-03,Male,39856.0,Hyperlipidemia (E78.5),17-06-2024


### 1c. Identify inconsistent entries in 'Gender' column

In [559]:
# List all the unique input in column 'Gender'
df["Gender"].unique()

array(['Female', 'Male', 'M', 'male', 'F'], dtype=object)

### 1d. Identify inconsitent date  formats in the 'AdmissionDate' Column

In [560]:
# Check datatype
DataType = df["AdmissionDate"].dtype
print ('DataType =', DataType, '\n')

# Force a specific expected datetime format: mm-dd-yyy
converted_dates = pd.to_datetime(
    df["AdmissionDate"],
    format="%m-%d-%Y",
    errors="coerce"
)

# Find rows that failed conversion
invalid_rows = df[converted_dates.isna()]
print(invalid_rows)


DataType = object 

   PatientID DateOfBirth  Gender  ZipCode        PrimaryCondition  \
1      NV004         NaN    Male  76999.0        Asthma (J45.909)   
2      NV005  1984-08-25  Female  40307.0  Osteoarthritis (M15.9)   
4      NV007  1990-10-11    Male      NaN  Hyperlipidemia (E78.5)   
5      NV008  1949-07-08    Male  56844.0  Hyperlipidemia (E78.5)   
7      NV010  1944-11-11  Female  41617.0  Osteoarthritis (M15.9)   
8      NV012  1975-01-14  Female  33023.0      Hypertension (I10)   
10     NV015  1959-11-20       M  45116.0      Hypertension (I10)   
11     NV017  1970-07-06  Female  67793.0  Hyperlipidemia (E78.5)   
12     NV019  1944-08-03  Female  40307.0        Asthma (J45.909)   
14     NV022  1957-01-02  Female  45989.0      Hypertension (I10)   
15     NV023  1974-05-18    Male  62729.0  Osteoarthritis (M15.9)   
16     NV024  1958-08-13  Female  19045.0   Type 2 Diabetes (E11)   
17     NV025  1972-05-02  Female  54291.0   Type 2 Diabetes (E11)   
18     NV026  

1d Output Note: The date appeared into 3 format: mm-dd-yyy; dd-mm-yyy; and excel seriel date number (Row5 and Row 37)

###1e. Identify invalid 'PatientID', if any.

In [561]:
# Validity of 'PaitentID'
import re

pattern = r"^[A-Za-z]{2}\d{3}$"  #PatientID is consist of 2 leading Alpabert follow by 3 number

# Remove leading/trailing spaces first (important)
patient_ids = df["PatientID"].astype(str).str.strip()

# Check valid format
valid_mask = patient_ids.str.match(pattern)

# Rows that are invalid
invalid_ids = df[~valid_mask]

# Show results
print("Number of invalid PatientIDs:", len(invalid_ids))
display(invalid_ids)


Number of invalid PatientIDs: 0


,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate


###1f. Identify inconsitent date formats in the 'DateOfBirth' Column, if any.

In [562]:
# Convery "DateOfBirth" to datetime.
df["DateOfBirth"] = pd.to_datetime(df["DateOfBirth"], errors="coerce")

# Check if the 'DateOfBirth' follow the format yyyy-mm-dd
invalid_dob = df[
    df["DateOfBirth"].dt.strftime("%Y-%m-%d") != df["DateOfBirth"].astype(str).str[:10]
]

print("Number of invalid DateOfBirth:", len(invalid_dob))
display(invalid_dob)

Number of invalid DateOfBirth: 6


,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
1,NV004,NaT,Male,76999.0,Asthma (J45.909),27-10-2022
20,NV028,NaT,Male,NaN,Type 2 Diabetes (E11),27-10-2022
21,NV028,NaT,Male,NaN,Type 2 Diabetes (E11),27-10-2022
24,NV031,NaT,Female,26647.0,Asthma (J45.909),07-08-2024
47,NV059,NaT,Female,53017.0,UNKNOWN,15-12-2023
67,NV088,NaT,Female,37775.0,Osteoarthritis (M15.9),02-09-2022


### 1f. Identify invalid 'Zipcode', if any.

In [563]:
# Convert ZipCode safely
zip_clean = df["ZipCode"].astype("Int64").astype(str)

# Valid ZIPs = exactly 5 digits
valid_mask = zip_clean.str.match(r"^\d{5}$")

# Invalid rows
invalid_zipcodes = df[~valid_mask]

print("Number of invalid ZIP codes:", len(invalid_zipcodes))

display(invalid_zipcodes)

Number of invalid ZIP codes: 4


,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
4,NV007,1990-10-11,Male,NaN,Hyperlipidemia (E78.5),17-06-2024
20,NV028,NaT,Male,NaN,Type 2 Diabetes (E11),27-10-2022
21,NV028,NaT,Male,NaN,Type 2 Diabetes (E11),27-10-2022
33,NV043,1970-07-08,Male,NaN,Hypertension (I10),09-03-2024


###2a. Handle Missing Values

In [564]:
# Remove row with missing values
df = df.dropna()

print(df.shape)

(66, 6)


###2b. Remove duplicate row


In [565]:
# drop duplicate
df = df.drop_duplicates()

print(df.shape)

(65, 6)


###2c. Standardize value in 'Gender'

In [566]:
# Stadarzide the value in Gender. Use "Male" and "Female" only
df["Gender"] = df["Gender"].replace({
    "M": "Male",
    "male": "Male",
    "F": "Female"
})

# Final check after standardization
# List all the unique input in column 'Gender'
df["Gender"].unique()


array(['Female', 'Male'], dtype=object)

### 2d. Standardize Date format in 'AdmissionDate'

In [567]:
# Create a function to clean the 'AdmissionDate'
def clean_dates(x):
    try:
        # Case 1: Excel serial number
        if isinstance(x, (int, float)) or str(x).replace('.', '', 1).isdigit():
            return pd.to_datetime(float(x), unit="D", origin="1899-12-30")

        # Case 2: normal string date
        return pd.to_datetime(x, errors="coerce", dayfirst=True)

    except:
        return pd.NaT

df["AdmissionDate"] = df["AdmissionDate"].apply(clean_dates)


# Standardize final format
df["AdmissionDate"] = df["AdmissionDate"].dt.strftime("%m-%d-%Y")

In [568]:
# Final check after standardization

# Check for any missing value/ failed to convert in AdmissionDate
Missing_AdminssionDate = df["AdmissionDate"].isna().sum()
print("Missing AdmissionDate Value =", Missing_AdminssionDate, "\n")

# checks if all values match mm-dd-yyyy style
date_style_check= df["AdmissionDate"].astype(str).str.match(r"\d{2}-\d{2}-\d{4}").all()
print("All AdmissionDate Value Matches mm-dd-yyyy? --", date_style_check)

Missing AdmissionDate Value = 0 

All AdmissionDate Value Matches mm-dd-yyyy? -- True


##**Analyze Data Sets using SQL**

### 3.Medical Data SQLite Exploration

In [569]:
db_url = "https://foundations-of-healthcare-data-analytics-4e579d.gitlab.io/labs/medical_data.db"  # replace
db_path = Path("medical_data.db")

In [570]:
response = requests.get(db_url)
response.raise_for_status()
db_path.write_bytes(response.content)
print("Downloaded to", db_path)

Downloaded to medical_data.db


In [571]:
conn = sqlite3.connect(db_path)
cur = conn.cursor()

In [572]:
def run_query(query: str):
    df = pd.read_sql_query(query, conn)
    display(df)

In [573]:
# Prepare df for sqlite query
df.to_sql(
    "patients_cleaned",   # table name in SQL
    conn,                 # SQLite connection
    if_exists="replace",  # overwrite old table
    index=False           # do not include Pandas index
)

conn.commit()

### Inspect the Table Structure for:
1. pharmacy_claims
2. lab_results
3. patients_cleaned

In [574]:
run_query("PRAGMA table_info(pharmacy_claims);")

,cid,name,type,notnull,dflt_value,pk
0,0,ClaimID,TEXT,0,None,0
1,1,PatientID,TEXT,0,None,0
2,2,Medication,TEXT,0,None,0
3,3,Dosage,TEXT,0,None,0
4,4,FillDate,TIMESTAMP,0,None,0
5,5,Payer,TEXT,0,None,0
6,6,ClinicID,TEXT,0,None,0
7,7,ChargeAmount,REAL,0,None,0
8,8,PaidAmount,REAL,0,None,0


In [575]:
run_query("PRAGMA table_info(lab_results);")

,cid,name,type,notnull,dflt_value,pk
0,0,PatientID,TEXT,0,None,0
1,1,LabTestID,TEXT,0,None,0
2,2,CollectionDate,TIMESTAMP,0,None,0
3,3,TestName,TEXT,0,None,0
4,4,TestResultValue,REAL,0,None,0
5,5,Units,TEXT,0,None,0
6,6,ReferenceRangeLow,REAL,0,None,0
7,7,ReferenceRangeHigh,REAL,0,None,0
8,8,AbnormalFlag,TEXT,0,None,0


In [576]:
run_query("PRAGMA table_info(patients_cleaned);")

,cid,name,type,notnull,dflt_value,pk
0,0,PatientID,TEXT,0,None,0
1,1,DateOfBirth,TIMESTAMP,0,None,0
2,2,Gender,TEXT,0,None,0
3,3,ZipCode,REAL,0,None,0
4,4,PrimaryCondition,TEXT,0,None,0
5,5,AdmissionDate,TEXT,0,None,0


### Inspect the First 5 Rows of each table

In [577]:
run_query("SELECT * FROM pharmacy_claims LIMIT 5;")

,ClaimID,PatientID,Medication,Dosage,FillDate,Payer,ClinicID,ChargeAmount,PaidAmount
0,RX9024,NV001,Amlodipine,10mg,2024-10-18 00:00:00,Medicaid,C03,391.20,380.08
1,RX9059,NV001,Atorvastatin,10mg,2024-06-15 00:00:00,Medicaid,C03,365.17,252.57
2,RX9099,NV004,Lisinopril,20mg,2024-09-07 00:00:00,Private,C05,56.54,51.56
3,RX9002,NV005,Amlodipine,10mg,2024-01-22 00:00:00,Medicaid,C05,296.82,215.41
4,RX9032,NV005,Amlodipine,?,2024-12-05 00:00:00,Medicaid,C03,371.18,349.95


In [578]:
run_query("SELECT * FROM lab_results LIMIT 5;")

,PatientID,LabTestID,CollectionDate,TestName,TestResultValue,Units,ReferenceRangeLow,ReferenceRangeHigh,AbnormalFlag
0,NV094,LB0001,2024-12-01 00:00:00,HbA1c,9.22,%,4.0,6.0,High
1,NV094,LB0002,2024-01-21 00:00:00,Creatinine,1.22,mg/dL,0.6,1.3,Normal
2,NV094,LB0003,2023-08-11 00:00:00,HbA1c,7.83,%,4.0,6.0,High
3,NV027,LB0004,2022-06-19 00:00:00,HbA1c,7.24,%,4.0,6.0,High
4,NV027,LB0005,2024-08-03 00:00:00,HbA1c,4.13,%,4.0,6.0,Normal


In [579]:
run_query("SELECT * FROM patients_cleaned LIMIT 5;")

,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
0,NV001,1983-03-11 00:00:00,Female,30933.0,Type 2 Diabetes (E11),06-07-2024
1,NV005,1984-08-25 00:00:00,Female,40307.0,Osteoarthritis (M15.9),10-30-2024
2,NV006,1982-12-16 00:00:00,Female,56844.0,Type 2 Diabetes (E11),07-05-2022
3,NV008,1949-07-08 00:00:00,Male,56844.0,Hyperlipidemia (E78.5),11-28-2023
4,NV009,1955-07-11 00:00:00,Female,68045.0,Hyperlipidemia (E78.5),05-09-2024


### 3a: List all patients born before 1980

In [580]:
run_query("""
SELECT *
FROM patients_cleaned
WHERE DateOfBirth < '1980-01-01';
""")

,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
0,NV008,1949-07-08 00:00:00,Male,56844.0,Hyperlipidemia (E78.5),11-28-2023
1,NV009,1955-07-11 00:00:00,Female,68045.0,Hyperlipidemia (E78.5),05-09-2024
2,NV010,1944-11-11 00:00:00,Female,41617.0,Osteoarthritis (M15.9),06-25-2023
3,NV012,1975-01-14 00:00:00,Female,33023.0,Hypertension (I10),09-21-2024
4,NV014,1943-08-25 00:00:00,Female,26372.0,Hyperlipidemia (E78.5),07-07-2022
5,NV015,1959-11-20 00:00:00,Male,45116.0,Hypertension (I10),05-23-2023
6,NV017,1970-07-06 00:00:00,Female,67793.0,Hyperlipidemia (E78.5),07-16-2023
7,NV019,1944-08-03 00:00:00,Female,40307.0,Asthma (J45.909),03-19-2022
8,NV020,1964-01-20 00:00:00,Male,37525.0,UNKNOWN,04-03-2024
9,NV022,1957-01-02 00:00:00,Female,45989.0,Hypertension (I10),11-14-2024


###3b.List all female patiens with Type 2 Diabetes

In [581]:
run_query("""
SELECT *
FROM patients_cleaned
WHERE PrimaryCondition = 'Type 2 Diabetes (E11)' and Gender = 'Female';
""")

,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
0,NV001,1983-03-11 00:00:00,Female,30933.0,Type 2 Diabetes (E11),06-07-2024
1,NV006,1982-12-16 00:00:00,Female,56844.0,Type 2 Diabetes (E11),07-05-2022
2,NV024,1958-08-13 00:00:00,Female,19045.0,Type 2 Diabetes (E11),03-21-2024
3,NV025,1972-05-02 00:00:00,Female,54291.0,Type 2 Diabetes (E11),10-18-2022
4,NV026,1975-04-09 00:00:00,Female,99104.0,Type 2 Diabetes (E11),09-17-2022
5,NV034,1974-05-18 00:00:00,Female,14441.0,Type 2 Diabetes (E11),08-16-2023
6,NV051,1971-08-06 00:00:00,Female,29205.0,Type 2 Diabetes (E11),04-18-2022
7,NV070,1951-02-03 00:00:00,Female,17401.0,Type 2 Diabetes (E11),08-15-2022
8,NV072,1992-11-01 00:00:00,Female,12050.0,Type 2 Diabetes (E11),09-20-2023
9,NV074,1959-05-27 00:00:00,Female,95982.0,Type 2 Diabetes (E11),03-11-2023


###3c. Display all lab tests performed. List each patient (PatientID, Gender, and DateOfBirth from *patients*) with their Lab test name and result value (TestName and TestResultValue from *lab_results*). -- Use JOIN

In [582]:
run_query("""
SELECT p.PatientID, p.Gender, p.DateOfBirth, ls.TestName, ls.TestResultValue
FROM patients_cleaned p
JOIN lab_results ls ON p.PatientID = ls.PatientID
""")

,PatientID,Gender,DateOfBirth,TestName,TestResultValue
0,NV001,Female,1983-03-11 00:00:00,Creatinine,1.15
1,NV001,Female,1983-03-11 00:00:00,HbA1c,5.19
2,NV001,Female,1983-03-11 00:00:00,LDL Cholesterol,110.22
3,NV005,Female,1984-08-25 00:00:00,Creatinine,0.49
4,NV005,Female,1984-08-25 00:00:00,Creatinine,1.42
...,...,...,...,...,...
157,NV098,Male,1981-06-13 00:00:00,HbA1c,5.29
158,NV098,Male,1981-06-13 00:00:00,HbA1c,5.85
159,NV098,Male,1981-06-13 00:00:00,HbA1c,5.92
160,NV098,Male,1981-06-13 00:00:00,LDL Cholesterol,124.93


3d. Display all distinct patients with their payer(s). List each patient (PatientID, Gender, and DateofBirth from *patients*) with their payer(s) from *pharmacy_claims*).

In [583]:
run_query("""
SELECT p.PatientID, p.Gender, p.DateOfBirth, pc.Payer
FROM patients_cleaned p
JOIN pharmacy_claims pc ON p.PatientID = pc.PatientID
""")

,PatientID,Gender,DateOfBirth,Payer
0,NV001,Female,1983-03-11 00:00:00,Medicaid
1,NV001,Female,1983-03-11 00:00:00,Medicaid
2,NV005,Female,1984-08-25 00:00:00,Medicaid
3,NV005,Female,1984-08-25 00:00:00,Medicaid
4,NV006,Female,1982-12-16 00:00:00,Private
...,...,...,...,...
75,NV094,Female,1962-11-05 00:00:00,Medicare
76,NV096,Male,1972-01-04 00:00:00,Medicaid
77,NV098,Male,1981-06-13 00:00:00,Medicaid
78,NV098,Male,1981-06-13 00:00:00,Medicaid


**Disclaimer:**

This project is based on the final assignment from the Coursera course "Foundations of Healthcare Data Analytics" offered by SkillUp. The dataset is fictional and provided by SkillUp for educational purposes only. All data cleaning, analysis, and SQL work were independently extended by the author beyond the course requirements. This project is for educational and portfolio use only and does not involve real patient data or clinical decision-making.

###**End of Project**